# Vs3/Vl3 3/3 density + moving potential player

只做一件事：读取成功案例的 `snapshots.csv`，在 Jupyter 里用播放按钮和进度条查看 density 演化，同时在同一张图的右轴显示随 \(\phi\) 滑动的势阱。不生成 mp4/gif，不写额外文件。

In [16]:
from pathlib import Path
import os
from io import BytesIO

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-ecg1d')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
import ipywidgets as widgets
from IPython.display import display

RUN_REL = Path('out/pump_vs3pad_gapadaptive_T160pi/a8p000_K16_tmax502p655_VsER3p000_VlER3p000')

def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / RUN_REL / 'snapshots.csv').exists():
            return path
    raise FileNotFoundError('找不到成功案例 snapshots.csv；请从 1D_thouless_pumping 目录或本 notebook 目录启动 Jupyter。')

PROJECT_ROOT = find_project_root()
RUN_DIR = PROJECT_ROOT / RUN_REL
SNAPSHOT_CSV = RUN_DIR / 'snapshots.csv'
TRACE_CSV = RUN_DIR / 'trace.csv'

# 每隔多少个 saved snapshot 取一帧。改小更细，改大更快。
FRAME_STRIDE = 50
X_POINTS = 600

# 和 pumpconfig/legacy_prb_3_3.cpp 保持一致。
LATTICE_A = 8.0
VS_OVER_ER = 3.0
VL_OVER_ER = 3.0

print(SNAPSHOT_CSV)

/home/gyqyan/zexuan/ecg1d_code/1D_thouless_pumping/out/pump_vs3pad_gapadaptive_T160pi/a8p000_K16_tmax502p655_VsER3p000_VlER3p000/snapshots.csv


In [17]:
snapshots = pd.read_csv(SNAPSHOT_CSV)
trace = pd.read_csv(TRACE_CSV)
groups = [(int(sid), frame.reset_index(drop=True)) for sid, frame in snapshots.groupby('snapshot', sort=True)]
frame_indices = np.arange(0, len(groups), FRAME_STRIDE, dtype=int)

x_mean = trace['x_mean'].to_numpy(float)
x2 = trace['x2'].to_numpy(float)
sigma = np.sqrt(np.maximum(x2 - x_mean * x_mean, 0.0))
x = np.linspace(np.nanmin(x_mean - 4.0 * sigma) - 2.0,
                np.nanmax(x_mean + 4.0 * sigma) + 2.0,
                X_POINTS)
trace_t = trace['t'].to_numpy(float)
integrate = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

def density_from_snapshot(frame):
    u = frame['u_re'].to_numpy(float) + 1j * frame['u_im'].to_numpy(float)
    A = frame['A_re'].to_numpy(float) + 1j * frame['A_im'].to_numpy(float)
    B = frame['B_re'].to_numpy(float) + 1j * frame['B_im'].to_numpy(float)
    R = frame['R_re'].to_numpy(float) + 1j * frame['R_im'].to_numpy(float)
    xx = x[None, :]
    exponent = -(A + B)[:, None] * xx * xx + (2.0 * B * R)[:, None] * xx - (B * R * R)[:, None]
    psi = (u[:, None] * np.exp(exponent - np.nanmax(np.real(exponent)))).sum(axis=0)
    rho = np.abs(psi) ** 2
    area = integrate(rho, x)
    return rho / area if np.isfinite(area) and area > 0 else rho

def potential_over_Er(phi):
    return (-VS_OVER_ER * np.cos(4.0 * np.pi * x / LATTICE_A)
            -VL_OVER_ER * np.cos(2.0 * np.pi * x / LATTICE_A + phi))

density_cache = {}

def cached_density(frame_id):
    if frame_id not in density_cache:
        source_index = int(frame_indices[frame_id])
        density_cache[frame_id] = density_from_snapshot(groups[source_index][1])
    return density_cache[frame_id]

print(f'loaded {len(groups)} snapshots; player frames = {len(frame_indices)}')

loaded 10055 snapshots; player frames = 202


In [18]:
play = widgets.Play(value=0, min=0, max=len(frame_indices) - 1, step=1, interval=160)
slider = widgets.IntSlider(value=0, min=0, max=len(frame_indices) - 1, step=1, continuous_update=False, description='frame')
widgets.jslink((play, 'value'), (slider, 'value'))
status = widgets.HTML()

fig = Figure(figsize=(9.5, 4.9), dpi=115)
canvas = FigureCanvasAgg(fig)
ax_rho = fig.add_subplot(111)
ax_v = ax_rho.twinx()

rho0 = cached_density(0)
snapshot_id0, frame0 = groups[int(frame_indices[0])]
phi0 = float(frame0['phi'].iloc[0])
V0 = potential_over_Er(phi0)
tr0 = trace.iloc[int(np.argmin(np.abs(trace_t - float(frame0['t'].iloc[0]))))]

rho_line, = ax_rho.plot(x, rho0, lw=1.9, color='#1f5f99', label='density')
pot_line, = ax_v.plot(x, V0, lw=1.5, color='#9b6a12', alpha=0.85, label='V / E_R')
center_line = ax_rho.axvline(float(tr0['x_mean']), color='black', lw=1.0, alpha=0.42)

ax_rho.set_xlim(x[0], x[-1])
ax_rho.set_ylim(0, 1.08 * float(np.nanmax(rho0)))
ax_v.set_ylim(-1.05 * (VS_OVER_ER + VL_OVER_ER), 1.05 * (VS_OVER_ER + VL_OVER_ER))
ax_rho.set_xlabel('x')
ax_rho.set_ylabel('normalized density', color='#1f5f99')
ax_v.set_ylabel('V / E_R', color='#9b6a12')
ax_rho.tick_params(axis='y', labelcolor='#1f5f99')
ax_v.tick_params(axis='y', labelcolor='#9b6a12')
title = ax_rho.set_title('')
fig.tight_layout()
image = widgets.Image(format='png', width=920)
png_cache = {}

def render_png(frame_id):
    frame_id = int(frame_id)
    if frame_id in png_cache:
        return png_cache[frame_id]
    source_index = int(frame_indices[frame_id])
    snapshot_id, frame = groups[source_index]
    t = float(frame['t'].iloc[0])
    tr = trace.iloc[int(np.argmin(np.abs(trace_t - t)))]
    phi = float(frame['phi'].iloc[0])
    rho = cached_density(frame_id)
    rho_line.set_ydata(rho)
    pot_line.set_ydata(potential_over_Er(phi))
    center_line.set_xdata([float(tr['x_mean']), float(tr['x_mean'])])
    if float(np.nanmax(rho)) > 0.92 * ax_rho.get_ylim()[1]:
        ax_rho.set_ylim(0, 1.08 * float(np.nanmax(rho)))
    title.set_text(f"snapshot={snapshot_id}, t={t:.3f}, phi={phi:.3f}, P={float(tr['polarization_cell']):.6f}")
    canvas.draw()
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=115, bbox_inches='tight')
    data = buf.getvalue()
    if len(png_cache) < 80:
        png_cache[frame_id] = data
    return data

def show(frame_id):
    image.value = render_png(frame_id)
    status.value = f"frame {int(frame_id) + 1}/{len(frame_indices)} &nbsp; cached density={len(density_cache)} &nbsp; cached image={len(png_cache)}"

def on_slider_change(change):
    if change['name'] == 'value':
        show(change['new'])

slider.observe(on_slider_change, names='value')
show(0)
display(widgets.VBox([widgets.HBox([play, slider]), status, image]))